# 02 — Sentiment Analysis
**BUSI 1783 Individual Project — Jay Panchal (001495232)**

This notebook scores every Reddit post for sentiment using VADER, then aggregates the post-level scores to a daily series. Each post's title and body are combined into one text and scored with VADER's compound score (bounded between −1 and +1). Removed or deleted bodies fall back to the title alone. Post-level scores are then averaged within each calendar day to produce the daily mean sentiment, alongside daily post volume and the daily standard deviation of sentiment (a measure of how divided opinion was that day).

## 2.1 Score every post with VADER
This loops over the full archive, scores each post, and keeps the date, compound sentiment and post score. Large runs print progress every 50,000 posts.

In [1]:
import json
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from datetime import datetime

filepath = r'C:\Users\asus\BUSI1783-Project\data\raw\r_Bitcoin_posts.jsonl'

# Initialise VADER
analyzer = SentimentIntensityAnalyzer()

posts = []
print("Loading and scoring posts... this may take a few minutes")

with open(filepath, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        try:
            post = json.loads(line)
            
            # Combine title + body for sentiment (title always exists, body sometimes removed)
            title = post.get('title', '') or ''
            body = post.get('selftext', '') or ''
            
            # Skip if body is removed/deleted - use title only in that case
            if body in ['[removed]', '[deleted]']:
                body = ''
            
            text = (title + ' ' + body).strip()
            
            # Skip empty posts
            if len(text) < 3:
                continue
            
            # Get date from timestamp
            date = datetime.utcfromtimestamp(post['created_utc']).date()
            
            # Run VADER
            sentiment = analyzer.polarity_scores(text)['compound']
            
            posts.append({
                'date': date,
                'sentiment': sentiment,
                'score': post.get('score', 0)
            })
        except Exception as e:
            continue
        
        # Progress update every 50,000 posts
        if (i + 1) % 50000 == 0:
            print(f"  Processed {i + 1:,} posts...")

print(f"\n✅ Done! Scored {len(posts):,} posts")

# Convert to DataFrame
reddit_df = pd.DataFrame(posts)
reddit_df['date'] = pd.to_datetime(reddit_df['date'])

print(f"Date range: {reddit_df['date'].min().date()} to {reddit_df['date'].max().date()}")
print(reddit_df.head())

Loading and scoring posts... this may take a few minutes


C:\Users\asus\AppData\Local\Temp\ipykernel_24156\3420695959.py:34: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  date = datetime.utcfromtimestamp(post['created_utc']).date()


  Processed 50,000 posts...
  Processed 100,000 posts...
  Processed 150,000 posts...
  Processed 200,000 posts...
  Processed 250,000 posts...

✅ Done! Scored 295,336 posts
Date range: 2022-01-01 to 2025-12-30
        date  sentiment  score
0 2022-01-01     0.0000      3
1 2022-01-01     0.0000      0
2 2022-01-01     0.0000      0
3 2022-01-01     0.5775     11
4 2022-01-01    -0.4019      2


## 2.2 Aggregate to a daily series
Post-level scores are collapsed to one row per day: mean sentiment, number of posts (volume), and the standard deviation of sentiment.

In [2]:
# Aggregate posts to daily level
daily_sentiment = reddit_df.groupby('date').agg(
    sentiment_mean=('sentiment', 'mean'),       # average mood that day
    post_volume=('sentiment', 'count'),          # how many posts that day
    sentiment_std=('sentiment', 'std')           # how divided opinions were
).reset_index()

print(f"✅ Aggregated to {len(daily_sentiment)} days")
print(f"Date range: {daily_sentiment['date'].min().date()} to {daily_sentiment['date'].max().date()}")
print(f"\nAverage posts per day: {daily_sentiment['post_volume'].mean():.0f}")
print(f"Average daily sentiment: {daily_sentiment['sentiment_mean'].mean():.4f}")
print("\nSample:")
print(daily_sentiment.head(10))

✅ Aggregated to 1460 days
Date range: 2022-01-01 to 2025-12-30

Average posts per day: 202
Average daily sentiment: 0.1800

Sample:
        date  sentiment_mean  post_volume  sentiment_std
0 2022-01-01        0.204679          179       0.441192
1 2022-01-02        0.184101          210       0.389305
2 2022-01-03        0.185326          276       0.435887
3 2022-01-04        0.170266          251       0.378446
4 2022-01-05        0.187467          317       0.395769
5 2022-01-06        0.065492          379       0.437035
6 2022-01-07        0.085309          345       0.414647
7 2022-01-08        0.075975          310       0.409593
8 2022-01-09        0.197555          238       0.424563
9 2022-01-10        0.119009          295       0.389249


Save the daily sentiment series.

In [3]:
# Save daily sentiment data
daily_sentiment.to_csv(
    r'C:\Users\asus\BUSI1783-Project\data\cleaned\daily_sentiment.csv',
    index=False
)
print("✅ Daily sentiment saved!")

✅ Daily sentiment saved!


---
*End of sentiment analysis. The daily sentiment series is saved to `data/cleaned/daily_sentiment.csv`.*